# Computing the energy of water on IBM

In [5]:
from typing import Optional

from joblib import Parallel, delayed
from tqdm import tqdm

import numpy as np
import pickle

import cirq
import openfermion as of

import quimb as qu
import quimb.tensor as qtn
import torch

import qiskit
from qiskit import qasm3
from qiskit_aer import AerSimulator
import qiskit_ibm_runtime
from qiskit_ibm_runtime import SamplerV2 as Sampler

In [6]:
service = qiskit_ibm_runtime.QiskitRuntimeService(name="NERSC-US")

computer = qiskit_ibm_runtime.fake_provider.FakeFez()
sampler = Sampler(computer)

## Read in the Hamiltonian

In [7]:
# basis = "6-311g" # "sto-3g"  # "6-31g"

basis = "3h2o"

In [8]:
ls waters/3h2o/transpiled_circuits/3h2o_102_adaptiterations.qasm

waters/3h2o/transpiled_circuits/3h2o_102_adaptiterations.qasm


In [11]:
num_qubits = {
    "h2o": 12,
    "2h2o": 24,
    "3h2o": 36,
    "sto-3g": 14,
    "6-31g": 26,
    "6-311g": 38,
}
hamiltonian_basis = {
    "h2o": "waters/h2o/transpiled_circuits/h2o_hamiltonian_012_adaptiterations.pkl",
    "2h2o": "waters/2h2o/transpiled_circuits/2h2o_hamiltonian_143_adaptiterations.pkl",
    "3h2o": "waters/3h2o/transpiled_circuits/3h2o_hamiltonian_102_adaptiterations.pkl",
    "sto-3g": "37mHa/sto-3g/sto-3g_hamiltonian_001_adaptiterations.pkl",
    "6-31g": "37mHa/6-31g/6-31g_hamiltonian_007_adaptiterations.pkl",
    "6-311g": "37mHa/6-311g/6-311g_hamiltonian_253_adaptiterations.pkl"
}

circuit_basis = {
    "h2o": "waters/h2o/transpiled_circuits/h2o_012_adaptiterations.qasm",
    "2h2o": "waters/2h2o/transpiled_circuits/2h2o_143_adaptiterations.qasm",
    "3h2o": "waters/3h2o/transpiled_circuits/3h2o_102_adaptiterations.qasm",
    "sto-3g": "37mHa/sto-3g/sto-3g_001_adaptiterations.qasm",
    "6-31g": "37mHa/6-31g/6-31g_hamiltonian_007_adaptiterations.pkl",
    "6-311g": "37mHa/6-311g/6-311g_253_adaptiterations.qasm"
}


In [12]:
qubit_op = pickle.load(
    open(f"{hamiltonian_basis[basis]}", "rb")
)
of.utils.count_qubits(qubit_op)

36

In [13]:
# QubitOp from fcidump.
# import pyscf.tools
# from pyscf import ao2mo
# fcidump = pyscf.tools.fcidump.read(f"{basis}/{basis}.fcidump")

# n_orbitals = fcidump.get("NORB")
# num_electrons = fcidump.get("NELEC")
# ecore = fcidump.get("ECORE")
# h1 = fcidump.get("H1")
# h2 = fcidump.get("H2")
# h2 = ao2mo.restore(1, h2, n_orbitals)

# one_body_tensor, two_body_tensor = of.ops.representations.get_tensors_from_integrals(
#     h1, h2
# )

# qubit_op = of.jordan_wigner(
#         of.get_fermion_operator(
#             of.InteractionOperator(
#             constant=ecore,
#             one_body_tensor=one_body_tensor,
#             two_body_tensor=two_body_tensor,
#         )
#     )
# )

In [14]:
def convert_qubitop_to_mpo(qubit_op, n_qubits, max_bond=None, cutoff=1e-10, verbose: bool = False):
    mpo_list = []
    
    for i, (term, coeff) in enumerate(qubit_op.terms.items()):
        ops = [np.eye(2) for _ in range(n_qubits)]
        for idx, pauli_type in term:
            ops[idx] = qu.spin_operator(pauli_type)
        
        term_mpo = qtn.MPO_product_operator(ops) * coeff
        mpo_list.append(term_mpo)

    mpo = mpo_list[0]
    nterms = len(mpo_list)
    for i in range(1, len(mpo_list)):
        if verbose:
            print(f"\rOn term {i + 1} / {nterms}", end="")
        mpo += mpo_list[i]
        mpo.compress(max_bond=max_bond, cutoff=cutoff)
    
    return mpo

In [15]:
# # Parallel version.
# def add_pair(a, b, max_bond, cutoff):
#     a = a + b
#     a.compress(max_bond=max_bond, cutoff=cutoff)
#     return a


# def tree_sum(mpos, max_bond=None, cutoff=1e-10, n_jobs=-1):

#     total_adds = len(mpos) - 1
#     pbar = tqdm(total=total_adds)

#     while len(mpos) > 1:

#         pairs = [(mpos[i], mpos[i+1]) for i in range(0, len(mpos)-1, 2)]

#         results = Parallel(n_jobs=n_jobs, prefer="threads")(
#             delayed(add_pair)(a, b, max_bond, cutoff)
#             for a, b in pairs
#         )

#         pbar.update(len(pairs))

#         if len(mpos) % 2 == 1:
#             results.append(mpos[-1])

#         mpos = results

#     pbar.close()
#     return mpos[0]


# def build_term(term, coeff, n_qubits):
#     term_dict = dict(term)

#     ops = [
#         qu.spin_operator(term_dict[i]) if i in term_dict else np.eye(2)
#         for i in range(n_qubits)
#     ]

#     return qtn.MPO_product_operator(ops) * coeff


# def convert_qubitop_to_mpo(qubit_op, n_qubits, max_bond=None, cutoff=1e-10):
#     mpo_list = Parallel(n_jobs=-1, prefer="threads")(
#         delayed(build_term)(term, coeff, n_qubits)
#         for term, coeff in qubit_op.terms.items()
#     )
#     return tree_sum(mpo_list, max_bond, cutoff)


# mpo = convert_qubitop_to_mpo(qubit_op, n_qubits=num_qubits[basis], max_bond=16)

In [16]:
mpo = qu.load_from_disk('mpo_2h2o_64')

In [17]:
max_mpo_bond: int = 64

In [18]:
mpo = convert_qubitop_to_mpo(qubit_op, n_qubits=num_qubits[basis], max_bond=max_mpo_bond, verbose=True)

On term 6543 / 6543

In [19]:
qu.save_to_disk(mpo, f"mpo_{basis}_{max_mpo_bond}")

['mpo_3h2o_64']

In [20]:
mpo

MatrixProductOperator(tensors=36, indices=107, L=36, max_bond=34)

In [21]:
dmrg = qtn.DMRG2(mpo, bond_dims=[10, 20, 50, 100], cutoffs=1e-10)
dmrg.solve(max_sweeps=10, verbosity=1, tol=1e-8)
ground_state_energy = dmrg.energy
ground_state_mps = dmrg.state  # This is the optimized Matrix Product State (MPS)

print(f"Ground state energy: {ground_state_energy}")

1, R, max_bond=(10/10), cutoff:1e-10


100%|##########################################| 35/35 [00:00<00:00, 109.40it/s]

Energy: (-218.31954642962853-1.4566126083082054e-11j) ... not converged.
2, R, max_bond=(10/20), cutoff:1e-10



100%|###########################################| 35/35 [00:01<00:00, 17.78it/s]

Energy: (-218.43325434121274+3.154809746774845e-12j) ... not converged.
3, R, max_bond=(20/50), cutoff:1e-10


...

Energy: (-218.49187414269582-3.481659405224491e-12j) ... not converged.
8, R, max_bond=(16/100), cutoff:1e-10



100%|##########################################| 35/35 [00:00<00:00, 289.61it/s]

Energy: (-218.49187623463007-3.602451670303708e-12j) ... not converged.
9, R, max_bond=(7/100), cutoff:1e-10



100%|##########################################| 35/35 [00:00<00:00, 482.90it/s]

Energy: (-218.4918762407822-3.574029960873304e-12j) ... converged!
Ground state energy: (-218.4918762407822-3.574029960873304e-12j)


## Read in the circuit

In [22]:
base = qasm3.load(f"{circuit_basis[basis]}")
base = qiskit.transpiler.passes.RemoveBarriers()(base)
# base.draw(fold=-1, idle_wires=False)

In [23]:
base.count_ops()

OrderedDict([('cx', 4141),
             ('rz', 2713),
             ('sx', 847),
             ('s', 828),
             ('h', 653),
             ('x', 102)])

## Qiskit MPS simulator with hardware noise model

In [24]:
# sim = AerSimulator.from_backend(computer, method="matrix_product_state")

In [25]:
# to_run = base.copy()
# to_run.measure_all()
# to_run = [to_run]
# to_run = qiskit.transpile(
#     to_run,
#     computer,
# )
# to_run[0].save_matrix_product_state(label="mps")
# to_run[0].draw(fold=-1, idle_wires=False)

In [26]:
# result = sim.run(to_run[0], shots=10_000)

In [27]:
# counts = result.result().get_counts()

In [28]:
# counts

In [29]:
# mps_tensors, mps_singular_values = result.result().data()["mps"]

## Cirq noise model / Quimb MPS

In [30]:
def qasm_to_cirq(qc: qiskit.QuantumCircuit) -> cirq.Circuit:
    # Create Cirq qubits
    qubits = [cirq.LineQubit(i) for i in range(qc.num_qubits)][::-1]

    circuit = cirq.Circuit(cirq.I.on(q) for q in qubits)

    for instruction, qargs, _ in qc.data:
        name = instruction.name
        params = instruction.params
        q = [qubits[qc.qubits.index(qubit)] for qubit in qargs]

        if name == "h":
            circuit.append(cirq.H(q[0]))
        
        elif name == "s":
            circuit.append(cirq.S(q[0]))

        elif name == "x":
            circuit.append(cirq.X(q[0]))

        elif name == "y":
            circuit.append(cirq.Y(q[0]))

        elif name == "z":
            circuit.append(cirq.Z(q[0]))

        elif name == "cx":
            circuit.append(cirq.CNOT(q[0], q[1]))

        elif name == "cz":
            circuit.append(cirq.CZ(q[0], q[1]))

        elif name == "swap":
            circuit.append(cirq.SWAP(q[0], q[1]))

        elif name == "rx":
            circuit.append(cirq.rx(params[0])(q[0]))

        elif name == "ry":
            circuit.append(cirq.ry(params[0])(q[0]))

        elif name == "rz":
            circuit.append(cirq.rz(params[0])(q[0]))

        elif name == "sx":
                circuit.append(cirq.XPowGate(exponent=0.5)(q[0]))

        elif name == "barrier":
            continue
            # ignore barriers

        else:
            raise NotImplementedError(f"Gate not supported: {name}")

    return circuit

In [31]:
circuit = qasm_to_cirq(base)
# circuit

/tmp/ipykernel_28566/1257008696.py:7: DeprecationWarning: Treating CircuitInstruction as an iterable is deprecated legacy behavior since Qiskit 1.2, and will be removed in Qiskit 3.0. Instead, use the `operation`, `qubits` and `clbits` named attributes.
  for instruction, qargs, _ in qc.data:


In [32]:
def simulate(
    circuit: cirq.Circuit,
    verbose: bool = False,
    seed: Optional[int] = None,
    backend: str = "cpu",
    max_bond: Optional[int] = None,
    cutoff: float = 1e-10,
    ) -> qtn.MatrixProductState:
    rng = np.random.RandomState(seed)

    qubits_to_indices = {q: i for i, q in enumerate(sorted(circuit.all_qubits()))}
    nqubits = len(qubits_to_indices)

    mps = qtn.MPS_computational_state("0" * nqubits, dtype="float64", cyclic=False)

    if backend == "gpu":
        for tensor in mps.tensors:
            tensor.modify(
                apply=lambda x: torch.tensor(x, dtype=torch.complex64, device="cuda")
            )

    num_ops = len(list(circuit.all_operations()))
    for i, op in enumerate(circuit.all_operations()):
        qubit_indices = [qubits_to_indices[q] for q in op.qubits]
        if cirq.has_unitary(op):
            to_apply = qu.qarray(cirq.unitary(op))
        elif cirq.has_mixture(op):
            ps = []
            ops = []
            for (p, o) in cirq.mixture(op):
                ps.append(p)
                ops.append(o)
            op = ops[rng.choice(range(len(ops)), p=ps)]
            to_apply = qu.qarray(op)
        else:
            raise ValueError(f"Cannot apply operation {op}")
        
        if backend == "gpu":
            to_apply = torch.tensor(to_apply, dtype=torch.complex64, device="cuda")

        mps.gate_(
            to_apply,
            qubit_indices,
            contract="swap+split",
            max_bond=max_bond,
            cutoff=cutoff,
        )
        if verbose:
            print(f"\rOp {i + 1} / {num_ops}", end="")

    return mps

In [33]:
max_mps_bond: int = 16

In [34]:
mps = simulate(circuit, max_bond=max_mps_bond, backend="cpu", verbose=True)
mps

Op 9320 / 9320

MatrixProductState(tensors=36, indices=71, L=36, max_bond=15)

In [35]:
# If on GPU:
# for tensor in mpo.tensors:
#     tensor.modify(
#         apply=lambda x: torch.tensor(x, dtype=torch.complex64, device="cuda")
#     )

In [36]:
def quimb_overlap(mps: qtn.MatrixProductState, mpo: qtn.MatrixProductOperator) -> float:
    mpsH = mps.H
    mps.align_(mpo, mpsH)
    return (mpsH & mpo & mps) ^ ...

In [37]:
exact_energy = quimb_overlap(mps, mpo)
exact_energy

-208.07103547277467

In [38]:
noise_rates = [1e-8, 1e-6, 1e-4, 1e-2]
ntrajectories = 1_000

noisy_energies = []
noisy_energies_stds = []
errors = []
for noise_rate in noise_rates:
    print("Status: On noise rate", noise_rate)
    noisy = circuit.with_noise(cirq.depolarize(noise_rate))
    # print(noisy)

    all_mps = Parallel(n_jobs=-1)(
        delayed(simulate)(noisy, max_bond=max_mps_bond, backend="cpu")
        for _ in tqdm(range(ntrajectories), desc="Simulating trajectories")
    )
    overlaps = [quimb_overlap(mps, mpo) for mps in all_mps]
    noisy_energy = np.average(overlaps)
    noisy_energies.append(noisy_energy)
    noisy_energies_stds.append(np.std(noisy_energies, ddof=1))
    errors.append(np.abs(noisy_energy - exact_energy))
    print("Noisy energy:", noisy_energy)
    print("Exact energy:", exact_energy)
    print("Absolute error:", errors[-1])
    print("St.Dev. over trajectories:", noisy_energies_stds[-1])

Status: On noise rate 1e-08


Simulating trajectories: 100%|██████████| 1000/1000 [57:38<00:00,  3.46s/it]
/home/ryan/prof/work/wellcome/q4bio/envq4bio/lib/python3.10/site-packages/numpy/_core/_methods.py:223: RuntimeWarning: Degrees of freedom <= 0 for slice
  ret = _var(a, axis=axis, dtype=dtype, out=out, ddof=ddof,
/home/ryan/prof/work/wellcome/q4bio/envq4bio/lib/python3.10/site-packages/numpy/_core/_methods.py:215: RuntimeWarning: invalid value encountered in scalar divide
  ret = ret.dtype.type(ret / rcount)


Noisy energy: -208.07200226244922
Exact energy: -208.07103547277467
Absolute error: 0.0009667896745497728
St.Dev. over trajectories: nan
Status: On noise rate 1e-06


Simulating trajectories: 100%|██████████| 1000/1000 [57:33<00:00,  3.45s/it]


Noisy energy: -208.06919975816686
Exact energy: -208.07103547277467
Absolute error: 0.0018357146078074038
St.Dev. over trajectories: 0.0019816697823590982
Status: On noise rate 0.0001


Simulating trajectories: 100%|██████████| 1000/1000 [57:32<00:00,  3.45s/it]


Noisy energy: -206.6340827067239
Exact energy: -208.07103547277467
Absolute error: 1.43695276605078
St.Dev. over trajectories: 0.8293754129976317
Status: On noise rate 0.01


Simulating trajectories: 100%|██████████| 1000/1000 [57:14<00:00,  3.43s/it]


Noisy energy: -181.5144097834027
Exact energy: -208.07103547277467
Absolute error: 26.55662568937197
St.Dev. over trajectories: 13.05624926397218


In [39]:
# noisy

In [40]:
noisy_energies

[np.float64(-208.07200226244922),
 np.float64(-208.06919975816686),
 np.float64(-206.6340827067239),
 np.float64(-181.5144097834027)]

In [41]:
exact_energy

-208.07103547277467

In [42]:
errors

[np.float64(0.0009667896745497728),
 np.float64(0.0018357146078074038),
 np.float64(1.43695276605078),
 np.float64(26.55662568937197)]

In [43]:
noisy_energies_stds

[np.float64(nan),
 np.float64(0.0019816697823590982),
 np.float64(0.8293754129976317),
 np.float64(13.05624926397218)]

In [44]:
basis

'3h2o'

In [45]:
np.savetxt(f"{basis}_noiserates_errors_stds.txt", np.array([noise_rates, errors, noisy_energies_stds]).T)